---
title: "Лабораторна робота №3. Обчислення ентропійних характеристик інформаційної системи"
description: "Моніторинг та керування в слабкоструктурованих процесах і системах"
author: "&copy; Роменський В'ячеслав"
date: today
format:
  html:
    toc: true
    toc-depth: 4
    toc-location: right
    code-fold: false
    code-tools: true
    embed-resources: true
    smooth-scroll: true
jupyter: python3
---


## Мета роботи

Опанувати підходи до *кількісного оцінювання невизначеності* в інформаційних системах та навчитися:

- обчислювати *максимальну* і *реальну* ентропію дискретного джерела;
- порівнювати ентропії різних джерел інформації;
- знаходити *часткову* та *повну умовну ентропію*;
- обчислювати *взаємну інформацію* між випадковими величинами;
- інтерпретувати одержані результати у задачах *моніторингу станів системи*.


## 1. Початкові дані

In [16]:

# Базові імпорти
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

my_names = ["Норма", "Перегрів", "Аварія"]
my_px = np.array([0.7, 0.2, 0.1])

# Рядки - реальність, Стовпці - що показав прилад
my_p_y_given_x = np.array([
    [0.9,  0.08, 0.02],
    [0.15, 0.7,  0.15],
    [0.05, 0.1,  0.85]
])

## 2. Допоміжні функції

Нижче наведено універсальні функції для обчислення основних ентропійних характеристик.


In [17]:

def validate_probabilities(probabilities, tol=1e-9):
    probs = np.array(probabilities, dtype=float)
    if np.any(probs < 0):
        raise ValueError("Ймовірності не можуть бути від'ємними.")
    total = probs.sum()
    if not np.isclose(total, 1.0, atol=tol):
        raise ValueError(f"Сума ймовірностей має дорівнювати 1. Зараз: {total}")
    return probs

def hartley_entropy(m):
    return math.log2(m)

def shannon_entropy(probabilities):
    probs = validate_probabilities(probabilities)
    nonzero = probs[probs > 0]
    return -np.sum(nonzero * np.log2(nonzero))

def normalized_entropy(probabilities):
    probs = validate_probabilities(probabilities)
    h = shannon_entropy(probs)
    h_max = hartley_entropy(len(probs))
    return h / h_max if h_max > 0 else 0.0

def conditional_entropy_from_conditional(px, p_y_given_x):
    px = validate_probabilities(px)
    cond = np.array(p_y_given_x, dtype=float)
    if cond.shape[0] != len(px):
        raise ValueError("Кількість рядків матриці умовних ймовірностей має збігатися з розміром p(x).")
    for row in cond:
        validate_probabilities(row)
    value = 0.0
    partial = []
    for i, row in enumerate(cond):
        h_row = shannon_entropy(row)
        partial.append(h_row)
        value += px[i] * h_row
    return value, np.array(partial)

def joint_distribution(px, p_y_given_x):
    px = validate_probabilities(px)
    cond = np.array(p_y_given_x, dtype=float)
    joint = px[:, None] * cond
    return joint

def entropy_from_joint(joint):
    probs = np.array(joint, dtype=float).ravel()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def mutual_information(px, py, joint=None, p_y_given_x=None):
    px = validate_probabilities(px)
    py = validate_probabilities(py)
    hx = shannon_entropy(px)
    hy = shannon_entropy(py)
    if joint is None:
        if p_y_given_x is None:
            raise ValueError("Потрібно задати або joint, або p_y_given_x.")
        joint = joint_distribution(px, p_y_given_x)
    hxy = entropy_from_joint(joint)
    ixy = hx + hy - hxy
    return hx, hy, hxy, ixy


## 3. Обчисліть максимальну та реальну ентропію

Обрахуємо максимальну ентропію (тобто при умові, що всі стани рівномірні) та порівняємо, наскільки ентропія Шеннона менша за цю.


In [18]:
# 2. Обчисліть максимальну та реальну ентропію
my_h_max = hartley_entropy(len(my_px))
my_h = shannon_entropy(my_px)

print("Hmax =", round(my_h_max, 4))
print("H =", round(my_h, 4))

Hmax = 1.585
H = 1.1568


## 4. Часткова та повна умовна ентропія

Обрахуємо ентропію при настанні кожного із трьох станів. Із вихідної таблиці ми можемо зрозуміти, що датчики дуже погано працюють із станом "Перегрів" і найкраще із станом "Норма". Проте насправді найбільше шуму йде саме від стану "Норма", бо він найбільш ймовірний.
Повна умовна ймовірність вказує, що система втрачає 0.6898 біт інформації. Так як для повної визначеності в нашій системі необхідно мати 1.15 біт інформації, то втрата дуже суттєва.

In [25]:
my_h_y_given_x, my_partial = conditional_entropy_from_conditional(my_px, my_p_y_given_x)
partial_df = pd.DataFrame({
    "Стан X": my_names,
    "Часткова умовна ентропія H(Y|x_i)": my_partial,
    "Вага p(x_i)": my_px,
    "Внесок у повну H(Y|X)": my_partial * my_px
})
partial_df

,Стан X,Часткова умовна ентропія H(Y|x_i),Вага p(x_i),Внесок у повну H(Y|X)
0,Норма,0.541188,0.7,0.378832
1,Перегрів,1.181291,0.2,0.236258
2,Аварія,0.747585,0.1,0.074758


In [27]:
print(f"Повна умовна ентропія H(Y|X) = {my_h_y_given_x:.4f} біт")

Повна умовна ентропія H(Y|X) = 0.6898 біт


## 5. Спільні ймовірності

Створимо матрицю, що вказує на ймовірності одночасного настання події із масиву X та Y.

In [46]:

my_joint = joint_distribution(my_px, my_p_y_given_x)
my_py = my_joint.sum(axis=0)

joint_df = pd.DataFrame(
    my_joint_xy,
    index=my_names,
    columns=my_names
)

joint_df


,Норма,Перегрів,Аварія
Норма,0.630,0.056,0.014
Перегрів,0.030,0.140,0.030
Аварія,0.005,0.010,0.085


In [47]:
py_df = pd.DataFrame({
    "Стан Y": my_names,
    "Ймовірність": my_py
})

py_df

,Стан Y,Ймовірність
0,Норма,0.665
1,Перегрів,0.206
2,Аварія,0.129


## 6. Взаємна інформація

Тепер знайдемо ту інформацію, яку нам надають датчики (тобто від реальної ентропії мінус шум).

In [48]:
my_hx, my_hy, my_hxy, my_i = mutual_information(my_px, my_py, joint=my_joint)

print("I(X;Y) =", round(my_i, 4))

I(X;Y) = 0.5522


## Висновки

У ході лабораторної роботи було досліджено ентропійні характеристики інформаційної системи та реалізовано обчислення основних інформаційних показників: ентропії Хартлі та Шеннона, умовної ентропії, спільного розподілу ймовірностей та взаємної інформації. Отримані результати дозволили оцінити рівень невизначеності системи та кількість інформації, яку надають датчики моніторингу.

Було встановлено, що реальна ентропія системи є меншою за максимальну через нерівномірний розподіл станів. Аналіз умовної ентропії показав наявність інформаційних втрат і шуму в роботі датчиків, а значення взаємної інформації підтвердило, що показники приладів містять корисну інформацію про реальний стан системи.

Таким чином, ентропійні характеристики є ефективним інструментом для аналізу невизначеності, оцінювання якості інформації та підтримки процесів моніторингу й керування слабкоструктурованими системами.

## Контрольні питання

1. Що таке *кількість інформації* та *ентропія*?
Кількість інформації визначає, наскільки новим або несподіваним є повідомлення. Ентропія характеризує середній рівень невизначеності або інформації в системі.
2. У чому полягає відмінність між *формулою Хартлі* та *формулою Шеннона*?
Формула Хартлі використовується для рівноймовірних подій, а формула Шеннона враховує різні ймовірності появи подій.
3. Чому за нерівномірного розподілу ентропія зменшується?
Якщо деякі події більш ймовірні за інші, невизначеність системи зменшується, тому ентропія стає меншою.
4. Що показує *умовна ентропія*?
Умовна ентропія показує рівень невизначеності однієї величини за умови, що значення іншої вже відоме.
5. Що таке *часткова умовна ентропія* і чим вона відрізняється від повної?
Часткова умовна ентропія визначається для конкретного значення умови, а повна — як середнє значення для всіх можливих умов.
6. Що характеризує *взаємна інформація*?
Взаємна інформація показує, скільки інформації одна випадкова величина містить про іншу.
7. Коли взаємна інформація дорівнює нулю?
Взаємна інформація дорівнює нулю, коли випадкові величини є незалежними.
8. Як ентропійні характеристики можна використовувати у задачах *моніторингу та керування*?
Ентропійні характеристики застосовують для оцінки невизначеності, виявлення змін у системах та підвищення ефективності керування.